# PanTS SegResNet on Colab Pro — random vs SuPreM initialization

**This notebook orchestrates. It contains no science.** Every model, transform,
sampler, loss and checkpoint lives in `src/` in the pinned Git commit, so what
runs here is exactly what runs on the laptop.

The controlled ablation is two runs that differ in **one** argument:

| | arm A | arm B |
|---|---|---|
| name | SegResNet-Random | SegResNet initialized from SuPreM supervised pretraining |
| `--initialization` | `random` | `suprem` |
| everything else | identical | identical |

Prepared data, fold, seed, batch size, sampling, augmentation, DiceCE loss,
optimizer, schedule, AMP, step count and validation procedure are shared.

**Data flow.** Google Drive is *persistent transport*. `/content` is the
*ephemeral fast disk* training actually reads. Nothing trains from
`/content/drive`: a FUSE mount cannot sustain random reads of thousands of
files per epoch.

**Run order:** 1 → 8 once per session, then 9 (calibration) or 10 (production).
Section 11 is the disconnect-recovery path.

PanTS-te is never read here.

## 0. The only thing you edit

Pin the exact commit. A branch name would silently change what you ran.

In [ ]:
# The immutable tag for the production-ready code. A tag is reproducible in a
# way a branch name is not: `agent/local-preprocess-colab` moves with every
# commit, this does not. Replace with a raw SHA only for a different revision.
#
# segresnet-production-v1 is the FIRST revision that is correct for production:
# earlier tags seed the model after constructing it and select best.pt from a
# stochastic patch loss. Do not pin segresnet-colab-v1 for a real run.
PINNED_COMMIT = "segresnet-production-v1"
REPO_URL      = "https://github.com/sabinthapa100/pants_sabin.git"

DRIVE_DATA    = "/content/drive/MyDrive/PanTS_prepared/segresnet"
DRIVE_RUNS    = "/content/drive/MyDrive/PanTS_runs"

REPO_DIR      = "/content/pants_sabin"
PREPARED_ROOT = "/content/PanTS_prepared"
RUNS_LOCAL    = "/content/runs"
CHECKPOINT    = "/content/pretrained/supervised_suprem_segresnet_2100.pth"

# Independently measured from the official release; the notebook refuses to
# proceed if the download does not match.
SUPREM_SHA256 = "2db81dc05cd9ea7234ca75e921e53e32b8716dc4cba88a6710742bfc282589a3"
SUPREM_URL    = ("https://huggingface.co/MrGiovanni/SuPreM/resolve/main/"
                 "supervised_suprem_segresnet_2100.pth?download=true")
print("pinned to:", PINNED_COMMIT)

## 1. What runtime did Colab actually give us?

Colab assigns different GPUs and different disk sizes per session. Nothing
below assumes an A100 or a fixed `/content` size.

In [ ]:
!nvidia-smi
!free -h
!df -h /content

## 2. Mount Drive (transport and persistence only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_DATA), f"prepared data not found at {DRIVE_DATA}"
print(sorted(os.listdir(DRIVE_DATA)))

## 3. Clone the repository at the pinned commit

In [ ]:
import subprocess, os

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--force", PINNED_COMMIT], cwd=REPO_DIR, check=True)

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                      capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_DIR,
                       capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
assert not dirty, f"working tree is dirty:\n{dirty}"
os.chdir(REPO_DIR)

## 4. Dependencies

Colab ships PyTorch; only MONAI is normally missing. PyTorch is deliberately
not pinned — the runtime's build matches its own CUDA driver.

In [ ]:
!pip -q install "monai==1.5.1" nibabel

import platform, torch, monai
print(f"python {platform.python_version()}")
print(f"torch  {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
print(f"monai  {monai.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    {props.name}  {props.total_memory/1024**3:.1f} GiB VRAM")
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU.")

## 5. Disk gate — refuse to start what cannot finish

Required space is the extracted cache, plus one shard of staging headroom
(we delete each archive right after extracting it), plus checkpoints and a
safety margin. If `/content` is too small this cell stops.

It does **not** quietly switch to uint8, train from Drive, or use part of the
data. Those would change the experiment.

In [ ]:
import json, shutil, glob, os

meta = json.load(open(f"{DRIVE_DATA}/preprocessing.json"))
shards = sorted(glob.glob(f"{DRIVE_DATA}/shards/*.tar"))
assert shards, "no shards found in Drive"

cache_bytes   = sum(os.path.getsize(s) for s in shards)   # tar ~= sum of npz
largest_shard = max(os.path.getsize(s) for s in shards)
overhead      = 4 * 1024**3           # checkpoints, logs, pip, repo
margin        = 5 * 1024**3
required      = cache_bytes + largest_shard + overhead + margin
free          = shutil.disk_usage("/content").free

GB = 1024**3
print(f"prepared cache      {cache_bytes/GB:7.1f} GiB  ({meta['case_count']} cases, {len(shards)} shards)")
print(f"largest shard       {largest_shard/GB:7.1f} GiB  (staging headroom)")
print(f"checkpoints/logs    {overhead/GB:7.1f} GiB")
print(f"safety margin       {margin/GB:7.1f} GiB")
print(f"REQUIRED            {required/GB:7.1f} GiB")
print(f"free on /content    {free/GB:7.1f} GiB")

if free < required:
    raise SystemExit(
        f"STOP: /content has {free/GB:.1f} GiB but needs {required/GB:.1f} GiB.\n"
        "Use a runtime with a larger disk (Colab Pro high-RAM/A100 runtimes get more),\n"
        "or run this study on a machine with adequate local storage.\n"
        "Do NOT train from /content/drive and do NOT stage a subset."
    )
print("\nOK to stage.")
print(json.dumps(meta, indent=2))

## 6. Stage shards into `/content`, one at a time

For each shard: copy → verify SHA256 → extract → **delete the archive** →
next. Peak disk is therefore *cache + one shard*, not *cache + all archives*.

Re-running is safe: already-extracted shards are skipped.

In [ ]:
import hashlib, os, shutil, subprocess, tarfile, time

os.makedirs(PREPARED_ROOT, exist_ok=True)

sums = {}
for line in open(f"{DRIVE_DATA}/SHA256SUMS"):
    digest, name = line.split()
    sums[os.path.basename(name)] = digest

def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

for source in shards:
    name = os.path.basename(source)
    marker = f"{PREPARED_ROOT}/.staged_{name}"
    if os.path.exists(marker):
        print(f"{name}: already staged")
        continue

    started = time.time()
    local = f"/content/{name}"
    shutil.copyfile(source, local)

    digest = sha256(local)
    if digest != sums[name]:
        os.remove(local)
        raise SystemExit(f"STOP: checksum mismatch for {name}\n  {digest}\n  {sums[name]}")

    with tarfile.open(local) as archive:
        archive.extractall(PREPARED_ROOT)
    os.remove(local)                      # free the archive immediately
    open(marker, "w").close()

    free = shutil.disk_usage("/content").free / 1024**3
    print(f"{name}: verified, extracted, archive removed "
          f"({time.time()-started:5.1f}s, {free:.1f} GiB free)")

for extra in ("manifest.json", "preprocessing.json"):
    shutil.copyfile(f"{DRIVE_DATA}/{extra}", f"{PREPARED_ROOT}/{extra}")
print("\nstaged to", PREPARED_ROOT)

## 7. Verify the staged cache with the repository's own reader

In [ ]:
import sys, glob, json
import numpy as np
sys.path.insert(0, REPO_DIR)
from src.data.prepared import is_case_complete, read_prepared_case

meta = json.load(open(f"{PREPARED_ROOT}/preprocessing.json"))
cases = sorted(glob.glob(f"{PREPARED_ROOT}/cases/*.npz"))
print(f"cases staged: {len(cases)} (expected {meta['case_count']})")
assert len(cases) == meta["case_count"], "incomplete staging - re-run section 6"

ids = [os.path.basename(c)[:-4] for c in cases]
numbers = [int(i.split("_")[1]) for i in ids]
assert len(set(ids)) == len(ids), "duplicate case ids"
assert max(numbers) <= 9000, "a PanTS-te identifier is present"
assert min(os.path.getsize(c) for c in cases) > 0, "a zero-length case file"

for path in cases[:: max(1, len(cases) // 20)]:
    assert is_case_complete(path, deep=True), path
    image, label = read_prepared_case(path)
    assert image.dtype == np.float16 and label.dtype == np.uint8
    assert image.shape == label.shape
    values = image.astype(np.float32)
    assert np.isfinite(values).all() and 0.0 <= values.min() and values.max() <= 1.0
    assert label.max() <= 28

print(f"id range {min(numbers)}..{max(numbers)} — PanTS-tr only")
print("sampled deep validation passed; cache is usable")

## 8. SuPreM checkpoint — download, verify, and prove the transfer

Fetched from the official release rather than a personal copy. The hash was
measured independently; a mismatch stops the notebook.

The transfer report must show **81 of 81** transferable tensors loaded. Only
`conv_final.2.conv.{weight,bias}` stays random: it maps features to class
logits, and SuPreM predicted a different label set, so those two tensors have
the wrong shape and no meaning here.

In [ ]:
import hashlib, os, subprocess

os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
if not os.path.exists(CHECKPOINT):
    subprocess.run(["wget", "-q", "--show-progress", "-O", CHECKPOINT, SUPREM_URL], check=True)

digest = hashlib.sha256(open(CHECKPOINT, "rb").read()).hexdigest()
print(f"size   {os.path.getsize(CHECKPOINT):,} bytes")
print(f"sha256 {digest}")
if digest != SUPREM_SHA256:
    os.remove(CHECKPOINT)
    raise SystemExit(f"STOP: checkpoint hash mismatch, expected {SUPREM_SHA256}")
print("hash verified against the independently measured value\n")

# The report compares a checkpoint against a MODEL, so build one first.
from src.models.segresnet import build_segresnet, format_transfer_report, suprem_transfer_report

report = suprem_transfer_report(build_segresnet("random"), CHECKPOINT)
print(format_transfer_report(report))
assert len(report["transferable"]) == 81, \
    f"expected 81 transferred tensors, got {len(report['transferable'])}"
assert len(report["excluded_output_head"]) == 2, report["excluded_output_head"]
print("\n81/81 transferable tensors match; only the PanTS class head stays random.")

## 9. CALIBRATION — measure this GPU before spending hours on it

**This section is not production.** It deliberately uses
`--max-steps-per-epoch` to time a fixed number of steps. Production never does:
there, an epoch must consume the entire fold-0 training loader.

Correctness is already established on the laptop. What is unknown is the
throughput of whichever GPU Colab assigned, and therefore the wall-clock cost of
each candidate training budget.

Run both arms: the architecture is identical, but confirming that the two arms
have the same throughput is itself a check that nothing differs except the
weights.

Do **not** read scientific meaning into these losses.

In [ ]:
import json, time, subprocess, sys
import torch

sys.path.insert(0, REPO_DIR)
from src.training.trainer import SegResNetTrainer, TrainingConfig

manifest = json.load(open(f"{PREPARED_ROOT}/manifest.json"))
split = json.load(open(f"{REPO_DIR}/pants_cv_v1.json"))

# The exact production data configuration. Only the step cap is different.
CALIB_STEPS = 80
BATCH_SIZE, SAMPLES_PER_CASE, ACCUMULATION, WORKERS = 2, 2, 1, 2

calibration = {}
for arm, ckpt in (("random", None), ("suprem", CHECKPOINT)):
    config = TrainingConfig(
        experiment=f"calib_{arm}", initialization=arm, pretrained_checkpoint=ckpt,
        prepared_root=PREPARED_ROOT, fold=0, epochs=1,
        batch_size=BATCH_SIZE, samples_per_case=SAMPLES_PER_CASE,
        gradient_accumulation_steps=ACCUMULATION, num_workers=WORKERS,
        output_root=RUNS_LOCAL,
    )
    trainer = SegResNetTrainer(config, manifest=manifest, split=split)
    steps_per_epoch = len(trainer.train_loader)          # measured, not assumed

    trainer.model.train()
    loader = iter(trainer.train_loader)
    for _ in range(5):                                    # warm-up, excluded
        trainer.scaler.scale(trainer._forward_loss(next(loader))).backward()
    trainer.optimizer.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()

    wait = 0.0
    wall = time.time()
    for step in range(CALIB_STEPS):
        t0 = time.time(); batch = next(loader); wait += time.time() - t0
        loss = trainer._forward_loss(batch)
        trainer.scaler.scale(loss / ACCUMULATION).backward()
        if (step + 1) % ACCUMULATION == 0:
            trainer.scaler.step(trainer.optimizer); trainer.scaler.update()
            trainer.optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    wall = time.time() - wall

    t0 = time.time(); trainer.save(trainer.run_dir / "calib.pt", 0.0, 0)
    save_s = time.time() - t0
    ckpt_mb = (trainer.run_dir / "calib.pt").stat().st_size / 1e6

    t0 = time.time(); patch_loss = trainer.validate(); patch_s = time.time() - t0

    sample = trainer.monitoring_cases[:6]
    full = trainer.monitoring_cases
    trainer.monitoring_cases = sample
    t0 = time.time(); trainer.validate_volumes(); volume_s = (time.time() - t0) / len(sample)
    trainer.monitoring_cases = full

    calibration[arm] = {
        "steps_per_epoch": steps_per_epoch,
        "batch_shape": tuple(batch["image"].shape),
        "steps_per_s": CALIB_STEPS / wall,
        "patches_per_s": CALIB_STEPS * batch["image"].shape[0] / wall,
        "dataloader_wait_frac": wait / wall,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
        "checkpoint_mb": ckpt_mb,
        "checkpoint_save_s": save_s,
        "patch_validation_s": patch_s,
        "whole_volume_s_per_case": volume_s,
        "monitoring_cases": len(full),
    }
    print(f"--- {arm} ---")
    for key, value in calibration[arm].items():
        print(f"  {key:26s} {value}")
    print()
    del trainer; torch.cuda.empty_cache()

!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv
!free -h | head -2

In [ ]:
# Cost table for the candidate budgets, from the numbers just measured.
# Persist checkpoints where a dead runtime cannot take them with it.
import os, shutil, time

source = f"{RUNS_LOCAL}/calib_random/calib.pt"
os.makedirs(f"{DRIVE_RUNS}/_timing", exist_ok=True)
t0 = time.time(); shutil.copyfile(source, f"{DRIVE_RUNS}/_timing/_t.pt")
drive_copy_s = time.time() - t0
os.remove(f"{DRIVE_RUNS}/_timing/_t.pt")

arm = calibration["random"]
steps = arm["steps_per_epoch"]
patches_per_epoch = steps * arm["batch_shape"][0]
epoch_train_s = steps / arm["steps_per_s"]
select_s = arm["whole_volume_s_per_case"] * arm["monitoring_cases"]

VALIDATE_EVERY = 5
print(f"GPU delivers {arm['steps_per_s']:.2f} steps/s, {arm['patches_per_s']:.1f} patches/s")
print(f"dataloader wait {100*arm['dataloader_wait_frac']:.0f}%   peak VRAM {arm['peak_vram_gb']:.2f} GB")
print(f"checkpoint {arm['checkpoint_mb']:.0f} MB: save {arm['checkpoint_save_s']:.2f}s, "
      f"Drive copy {drive_copy_s:.2f}s")
print(f"\none full epoch = {steps} forward steps = {patches_per_epoch} patches "
      f"= {epoch_train_s/60:.1f} min")
print(f"patch validation {arm['patch_validation_s']:.0f}s/epoch (diagnostic)")
print(f"whole-volume selection {arm['whole_volume_s_per_case']:.1f}s/case "
      f"x {arm['monitoring_cases']} = {select_s/60:.0f} min, every {VALIDATE_EVERY} epochs\n")

print(f"{'budget':>10} {'updates':>10} {'patches':>12} {'train h':>9} {'valid h':>9} {'total h':>9}")
for epochs in (40, 60, 80):
    updates = steps * epochs // ACCUMULATION
    train_h = epochs * (epoch_train_s + arm["patch_validation_s"]) / 3600
    valid_h = (epochs // VALIDATE_EVERY) * select_s / 3600
    print(f"{epochs:>7} ep {updates:>10,} {epochs*patches_per_epoch:>12,} "
          f"{train_h:>9.1f} {valid_h:>9.1f} {train_h+valid_h:>9.1f}")
print("\nSame table applies to both arms: throughput differs by <1%.")
print("Report these numbers before starting production.")

## 10. PRODUCTION — fold 0, both arms

**Do not run until the budget from section 9 has been agreed.**

Differences from calibration, all deliberate:

* **no `--max-steps-per-epoch`.** An epoch consumes the entire fold-0 training
  loader. A step cap here would silently turn "40 epochs" into 40 partial
  passes over a biased prefix of the data.
* **`--accumulation 1`.** 2 cases × 2 samples = 4 patches per forward, so one
  optimizer update per step. SegResNet normalizes with GroupNorm, whose
  statistics are computed per sample, so unlike BatchNorm there is no reason to
  inflate the effective batch just for stable normalization.
* **`--persistent-output-root`.** The trainer copies each completed checkpoint
  to Drive itself, inside the training process. A separate copy cell would
  never run if the runtime died mid-training.
* **`--validate-every-epochs`.** Deterministic whole-volume validation on the
  fixed monitoring subset. This, and only this, writes `best.pt`.

`SHARED_PRODUCTION` is built once and passed to both arms. There is no second
place to edit, which is what keeps the ablation controlled.

In [ ]:
# Configuration of the reported production runs. 64 epochs x 3,600 optimizer
# updates = 230,400 updates, the same order as the nnU-Net baseline this study
# compares against (default schedule 250,000).
#
# provenance.json in each run directory records what a checkpoint was actually
# trained with; this cell is only the recipe.
EPOCHS = 64
assert EPOCHS, "agree the training budget from section 9 before running production"

SHARED_PRODUCTION = [
    "--prepared-root", PREPARED_ROOT,
    "--manifest", f"{PREPARED_ROOT}/manifest.json",
    "--split", f"{REPO_DIR}/pants_cv_v1.json",
    "--fold", "0",
    "--epochs", str(EPOCHS),
    # NO --max-steps-per-epoch: an epoch is the whole fold-0 training loader.
    "--batch-size", "2", "--samples-per-case", "2", "--accumulation", "1",
    "--learning-rate", "1e-4", "--weight-decay", "1e-5",
    # Execution parameter, but it seeds the per-worker crop and augmentation
    # RNG, so both arms must use the same value.
    "--num-workers", "4", "--seed", "317",
    "--save-every-epochs", "1",
    "--validate-every-epochs", "5",
    "--monitoring-negatives", "50",
    "--output-root", RUNS_LOCAL,
    "--persistent-output-root", DRIVE_RUNS,
]
assert "--max-steps-per-epoch" not in SHARED_PRODUCTION
assert "--limit-cases" not in SHARED_PRODUCTION

SHARED_STRING = " ".join(SHARED_PRODUCTION)
print(SHARED_STRING)

In [ ]:
# ARM A — SegResNet-Random
!cd {REPO_DIR} && python scripts/train_segresnet.py --initialization random --experiment segresnet_random {SHARED_STRING}

In [ ]:
# ARM B — SegResNet initialized from SuPreM supervised pretraining.
# Identical to arm A except --initialization and --pretrained-checkpoint.
!cd {REPO_DIR} && python scripts/train_segresnet.py --initialization suprem --experiment segresnet_suprem --pretrained-checkpoint {CHECKPOINT} {SHARED_STRING}

## 11. RESUME after a Colab disconnect

When the runtime dies, `/content` is destroyed — repo, staged cache, and any
checkpoint not yet copied to Drive. Drive survives.

Recovery: run sections **1 → 8** again in the fresh runtime (same pinned
commit, restage the data), copy the checkpoint back from Drive, then run the
cell below.

**`--epochs` must equal the original run's value.** The cosine schedule stored
in the checkpoint was built for that horizon; resuming with a different number
replays the original curve and the trainer will warn. A real interruption
changes nothing about the configuration.

In [ ]:
EXPERIMENT = "segresnet_random"   # or segresnet_suprem
ARM_ARGS   = ([] if EXPERIMENT.endswith("random")
              else ["--pretrained-checkpoint", CHECKPOINT])

import os, shutil, subprocess, torch
os.makedirs(f"{RUNS_LOCAL}/{EXPERIMENT}", exist_ok=True)
shutil.copyfile(f"{DRIVE_RUNS}/{EXPERIMENT}/latest.pt",
                f"{RUNS_LOCAL}/{EXPERIMENT}/latest.pt")

state = torch.load(f"{RUNS_LOCAL}/{EXPERIMENT}/latest.pt", map_location="cpu",
                   weights_only=False)
provenance = state["config"]
print(f"resuming after epoch {state['epoch']}, global step {state['global_step']}")
print(f"  best {provenance['selection_metric_name']} = {state['best_metric']:.4f} "
      f"at epoch {provenance['selection_epoch']}")
print(f"  monitoring subset {provenance['monitoring_subset_fingerprint']} "
      f"({provenance['monitoring_cases']} cases)")
print(f"  commit {state['git_commit']}")
print(f"  manifest {provenance['manifest_sha256'][:12]}  split {provenance['split_sha256'][:12]}")

# The interrupted run's own horizon. Reusing it is not optional: the cosine
# schedule inside the checkpoint was built for this number of epochs, so a
# different value would replay the original curve against a different total and
# silently alter the learning-rate trajectory of the rest of training.
original_epochs = provenance["config"]["epochs"]
assert original_epochs == EPOCHS, (
    f"this checkpoint was trained with --epochs {original_epochs}, but EPOCHS "
    f"is {EPOCHS}. Restore the original value before resuming."
)

subprocess.run(
    ["python", "scripts/train_segresnet.py",
     "--initialization", "random" if EXPERIMENT.endswith("random") else "suprem",
     "--experiment", EXPERIMENT,
     "--resume", f"{RUNS_LOCAL}/{EXPERIMENT}/latest.pt",
     *SHARED_PRODUCTION, *ARM_ARGS],
    check=True, cwd=REPO_DIR,
)

## 12. Inspect what has been persisted

The trainer already writes `latest.pt`, `best.pt` and the JSON records to Drive
as each epoch completes, so this section only *inspects*. There is nothing to
copy manually and nothing is lost if you never run it.

In [ ]:
import json, os, glob

for run in sorted(glob.glob(f"{DRIVE_RUNS}/*")):
    name = os.path.basename(run)
    if name.startswith("_"):
        continue
    print(f"{name}: {sorted(os.listdir(run))}")
    summary_path = f"{run}/summary.json"
    if os.path.exists(summary_path):
        s = json.load(open(summary_path))
        print(f"   {s['selection_metric']} = {s['best_selection_metric']} "
              f"at epoch {s['best_epoch']} over {s['monitoring_cases']} cases "
              f"(subset {s['monitoring_subset_fingerprint']})")
    history_path = f"{run}/history.json"
    if os.path.exists(history_path):
        for record in json.load(open(history_path)):
            line = (f"   epoch {record['epoch']:3d}  train {record['train_loss']:.4f}  "
                    f"patch {record['patch_val_loss_diagnostic']:.4f}  "
                    f"lr {record['learning_rate']:.2e}")
            if "selection" in record:
                line += f"  class-28 Dice {record['selection']['mean_dice_on_positive_cases']:.4f}"
            print(line)

---

### What survives a disconnect

| | survives | why |
|---|---|---|
| `MyDrive/PanTS_prepared/` | yes | Drive is persistent storage |
| `MyDrive/PanTS_runs/<exp>/latest.pt` | yes | mirrored by the trainer itself, inside the training process, as each epoch completes |
| `MyDrive/PanTS_runs/<exp>/best.pt` | yes | same, written on a deterministic selection improvement |
| `/content/PanTS_prepared/` | **no** | ephemeral VM disk — restage from shards |
| `/content/pants_sabin/` | **no** | re-clone at the same pinned commit |
| `/content/runs/` | **no** | the Drive mirror is the durable copy |
| GPU/optimizer state in memory | **no** | rebuilt from `latest.pt` |

Section 12 only *inspects*; nothing depends on it having been run. At most one
epoch of work is lost to a disconnect.

Evaluation happens on the laptop, from the prepared cache for fold-0
development metrics and from the original NIfTIs for the held-out test, after
the model and its postprocessing are frozen. See
`scripts/evaluate_segresnet.py`. The training cache plays no part in inference
on unseen scans — see `scripts/infer_segresnet.py`.